In [ ]:
import os
import subprocess
import shutil
import sys

os.environ["TRAINER_TELEMETRY"] = "0"
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

import os
import shutil

def silent_error(func):
    def wrapper(path, *args, **kwargs):
        try:
            return func(path, *args, **kwargs)
        except Exception:
            # We skip the error and keep going
            print(f"Windows Lock - Skipping: {path}")
            return None
    return wrapper

os.unlink = silent_error(os.unlink)
os.remove = silent_error(os.remove)
os.rmdir = silent_error(os.rmdir)
shutil.rmtree = silent_error(shutil.rmtree)

from TTS.tts.configs.glow_tts_config import GlowTTSConfig
from TTS.config.shared_configs import BaseDatasetConfig 
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.glow_tts import GlowTTS
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from trainer import Trainer, TrainerArgs


import trainer.trainer
# Bypass Git check StopIteration
Trainer.init_training = lambda self, *args, **kwargs: (args[2], {})
# Bypass log cleanup WinError
trainer.trainer.Trainer.cleanup_log_file = lambda self, *args, **kwargs: None


# 2. Define Paths
root_path = r"C:\TTS-glowtts"
dataset_path = os.path.join(root_path, "sinhala_dataset")
output_path = os.path.join(root_path, "output_sinhala")
pretrained_path = os.path.join(root_path, "Base_Model", "model_file.pth")
config_path = os.path.join(root_path, "Base_Model", "config.json")

# 3. Load Config & Setup Audio
config = GlowTTSConfig()
config.load_json(config_path)
ap = AudioProcessor(**config.audio)

# 4. Initialize Tokenizer
tokenizer, config = TTSTokenizer.init_from_config(config)

# 5. Load Data
dataset_config = BaseDatasetConfig(
    dataset_name="sinhala_dataset",
    path=dataset_path,
    meta_file_train="metadata_final_train.csv",
    meta_file_val="metadata_final_test.csv",
    formatter="ljspeech"
)
train_samples, eval_samples = load_tts_samples([dataset_config], eval_split=False)

if not train_samples:
    raise ValueError(f"No training samples found! Check your path: {dataset_path} and your metadata file.")
else:
    print(f" > Found {len(train_samples)} training samples.")

if not eval_samples:
    print(" > Warning: No evaluation samples found. Training might fail during eval step.")

# 6. Initialize Model
model = GlowTTS(config, ap, tokenizer=tokenizer, speaker_manager=None)

# 6.5 Configuration Overrides
config.output_path = output_path
config.dashboard_logger = "tensorboard"
config.project_name = "sinhala_ft"
config.batch_size = 4              
config.eval_batch_size = 2         
config.mixed_precision = True      
config.num_loader_workers = 0      
config.cudnn_benchmark = False
config.lr=1e-4


config.max_audio_config = {"max_duration": 10} 
config.optimizer_params = {
    "weight_decay": 1e-6,
    "betas": [0.9, 0.98],
    "eps": 1e-9
    }

config.run_name = "run"
config.save_step = 1000 
config.print_step = 10

# Force memory clearing before trainer starts
import torch
torch.cuda.empty_cache()

# 7. Setup Trainer
trainer = Trainer(
    TrainerArgs(restore_path=pretrained_path, skip_train_epoch=False),
    config,
    output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

# 8. Start Fine-tuning
trainer.fit()

c:\TTS-glowtts\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 > Found 4827 training samples.
 > Warning: No evaluation samples found. Training might fail during eval step.


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 12
 | > Num. of Torch Threads: 6
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=C:\TTS-glowtts\output_sinhala\run-March-14-2026_10+42AM-0000000
c:\TTS-glowtts\.venv\lib\site-packages\trainer\trainer.py:552: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
 > Restoring from model_file.pth ...
 > Restoring Model...
 > Partial model initialization...
 | > Layer missing in the model definition: decoder.flows.2.start.weight_g
 | > Layer missing in the model definition: decoder.flows.2.start.weight_v
 | > Layer missing in the model definition: decoder.flows.2.wn.in_layers.0.wei

: 